## Python Algorithmic Trading Bot

In [178]:
from datetime import datetime
import pandas as pd
import yfinance as yf
import time

In [179]:
# Base blueprint for all strategies
class MyTradingStrategy:
    def __init__(self,name):
        self.__name = name
        
    def generate_signal(self, price_data):
        print("This method should be overridden")
        return "Hold"

    @property
    def name(self):
        return self.__name

In [180]:
# return signal buy or sell - Actual trading logic
class MySmaTradingStrategy(MyTradingStrategy):
    def __init__(self,swindow, lwindow):
        self.__swindow = swindow
        self.__lwindow = lwindow

        super().__init__("My SMA Trading Strategy")
    
    def generate_signal(self, price_data):
        
        if len(price_data) < self.__lwindow:
            return "Hold"
        short_avg = sum(price_data[-self.__swindow:]) / self.__swindow
        long_avg = sum(price_data[-self.__lwindow:]) / self.__lwindow

        if short_avg > long_avg:
            return "Buy"

        elif short_avg < long_avg:
            return "Sell"

        else:
            return "Hold"

    @property
    def swindow(self):
        return self.__swindow

    @property
    def lwindow(self):
        return self.__lwindow

   

In [181]:
# it keeps the records of the trades - Store trade information
class MyTrade:
    def __init__(self, strategy_name,signal, quantity):
        self.__strategy_name = strategy_name
        self.__signal = signal
        self.__quantity = quantity
        self.__timestamp = datetime.now()
    
    def execute(self):
        print(f"Executed {self.__signal} trade with the strategy {self.__strategy_name} with Quantity: {self.__quantity} at the time {self.__timestamp}")
    
    @property
    def strategy_name(self):
        return self.__strategy_name

    @property
    def signal(self):
        return self.__signal

    @property
    def quantity(self):
        return self.__quantity
    
    @property
    def timestamp(self):
        return self.__timestamp


In [182]:
ObjStrategy = MySmaTradingStrategy(3,5)

strategy_name = ObjStrategy.name
signal = ObjStrategy.generate_signal([1,2,3,4,5])

trade = MyTrade(strategy_name,signal, 10)

In [183]:
## Mock Trading API - it intereacts with the exchange  - Fake broker/exchange  - Fake trading simulation
class MockTradingAPI:
    def __init__(self,balance):
        self.__balance = balance

    def place_order(self,trade, price):
        if trade.signal == "Buy" and \
              self.__balance >= trade.quantity * price:
            self.__balance -= trade.quantity * price
            print(f"Placed a Buy trade at the price: {price}, and Remaining balance:{self.__balance} ")
        elif trade.signal == "Sell":
            self.__balance += trade.quantity * price
            print(f"Placed a Sell trade at the price: {price}, and Remaining balance:{self.__balance} ")
        else:
            print("Insufficient balance")
    
    @property
    def balance(self):
        return self.__balance


In [184]:
# MySmaTradingStrategy object
#           ↓
# generate_signal()
#           ↓
# signal = "Buy"
#           ↓
# MyTrade object
# (stores signal + quantity)
#           ↓
# MockTradingAPI
# (executes trade)
#           ↓
# balance updated

In [185]:
# this class should able to integrate all other classes

# api =  pass object of MockTradingAPI into constructor
#strategy =  object of strategy class - buy , sell or hold
# symbol = on which kind of asset going to trade e.g apple, microsoft, nvidia, 
class MyTradingSystem:
    def __init__(self, api,strategy, symbol):
        self.__api = api
        self.__strategy = strategy
        self.__symbol = symbol 
        self.__price_data = []

    ## now fetch the live prices of yfinance
    # -------------------------------------------------
    # but for production  in replace of ydownload
    # # pseudo-code

    # def on_market_tick(data):

    #     current_price = data["ltp"]

    #     self.__price_data.append(current_price)

    #     signal = strategy.generate_signal(
    #         self.__price_data
    #     )

    #     if signal=="Buy":
    #         execute_trade()
# ---------------------------------

    def fetch_price_data(self):
        data = yf.download(
                        self.__symbol,
                        period="1d",
                        interval="1m"
                )
        if not data.empty:
            # price = float(data['Close'].iloc[-1])
            price = data['Close'].iloc[-1].item()
            self.__price_data.append(price)
            
            if len(self.__price_data) > self.__strategy.lwindow:
                self.__price_data.pop(0)
                
                # print(f"Symbol: {self.__symbol}")
                # print(f"Fetched price : {price:.2f}")
                # print(f"Stored prices: {self.__price_data}")
        else:
            print("No data fetched")

    def run(self):
        self.fetch_price_data()
        signal = self.__strategy.generate_signal(self.__price_data)

        print("\n----- Trading Info -----")

        print(f"Stock : {self.__symbol}")

        print(f"Current Price : {round(self.__price_data[-1],2)}")

        print(f"Previous Prices : {[round(p,2) for p in self.__price_data]}")

        print(f"Signal : {signal}")

        print(f"Balance : {self.__api.balance}")
        if signal in ["Sell", "Buy"]:
            trade = MyTrade(self.__strategy.name,signal,1)
            trade.execute()
            self.__api.place_order(trade,self.__price_data[-1])

@property
def api(self):
    return self.__api

@property
def strategy(self):
    return self.__strategy

@property
def symbol(self):
    return self.__symbol

@property
def price_data(self):
    return self.__price_data

In [186]:
# symbol = 'AAPL'
symbol = 'RELIANCE.NS'
api = MockTradingAPI(balance=10000)
strategy = MySmaTradingStrategy(3,5)
system = MyTradingSystem(api, strategy,symbol)

for __ in range(10):
    system.run()
    print(f"Remaning balance: {api.balance}")
    time.sleep(2)

[*********************100%***********************]  1 of 1 completed



----- Trading Info -----
Stock : RELIANCE.NS
Current Price : 1311.6
Previous Prices : [1311.6]
Signal : Hold
Balance : 10000
Remaning balance: 10000


[*********************100%***********************]  1 of 1 completed



----- Trading Info -----
Stock : RELIANCE.NS
Current Price : 1311.6
Previous Prices : [1311.6, 1311.6]
Signal : Hold
Balance : 10000
Remaning balance: 10000


[*********************100%***********************]  1 of 1 completed


...


----- Trading Info -----
Stock : RELIANCE.NS
Current Price : 1311.6
Previous Prices : [1311.6, 1311.6, 1311.6, 1311.6, 1311.6]
Signal : Hold
Balance : 10000
Remaning balance: 10000


[*********************100%***********************]  1 of 1 completed



----- Trading Info -----
Stock : RELIANCE.NS
Current Price : 1311.6
Previous Prices : [1311.6, 1311.6, 1311.6, 1311.6, 1311.6]
Signal : Hold
Balance : 10000
Remaning balance: 10000


[*********************100%***********************]  1 of 1 completed



----- Trading Info -----
Stock : RELIANCE.NS
Current Price : 1311.6
Previous Prices : [1311.6, 1311.6, 1311.6, 1311.6, 1311.6]
Signal : Hold
Balance : 10000
Remaning balance: 10000
